In [49]:
import sys
import numpy as np
import pandas as pd
sys.path.append("/cs/casmip/alina.ryabtsev/FewShotLearning/Lemonade")
from Lemonade.VirtualRichard import VirtualRichard
from glob import glob
from Lemonade.constants import *
import os
import nibabel as nib
from Lemonade.utils import postprocess_predictions, postprocess_predictions_per_slice
from Lemonade import utils
import re
from tqdm import tqdm

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
variability_th = 4

Evaluate for support_analysis_10 (clustering, kmeans=20 and K=10) 

In [4]:
# Path to the folder with the scans and the corresponding predictions
predictions_files = sorted(glob(os.path.join(LIVER_LESIONS_DATASET, "*support_analysis_10.nii.gz")))
pattern = re.compile(r".*\d+_support_analysis_10.nii.gz")
predictions_files = [path for path in predictions_files if pattern.search(path)]
scans = [path.replace("_support_analysis_10.nii.gz", "_scan.nii.gz") for path in predictions_files]
liver_masks = [path.replace("_support_analysis_10.nii.gz", "_liver.nii.gz") for path in predictions_files]

In [5]:
# Load the scans and the predictions
scans = [nib.load(scan).get_fdata() for scan in scans]
predictions = [nib.load(prediction).get_fdata() for prediction in predictions_files]

In [6]:
liver_masks = [nib.load(liver_mask).get_fdata().astype("float64") for liver_mask in liver_masks]

In [7]:
# Get only predictions within the liver
predictions = [prediction * liver_mask for prediction, liver_mask in zip(predictions, liver_masks)]
# postprocess the predictions
post_predictions = postprocess_predictions(predictions, save_postprocessed=True, predictions_affines=[nib.load(path).affine for path in predictions_files], predictions_filenames=predictions_files)

100%|██████████| 93/93 [01:59<00:00,  1.28s/it]


In [8]:
# Get the voxel volume
voxel_vols = [np.prod(nib.load(path).header.get_zooms()) for path in predictions_files]

In [9]:
# Get tumor that are bigger than 10 mm in diameter
post_predictions_big = [utils.mask_by_diameter(prediction, voxel_vol, 10)[1] for prediction, voxel_vol in zip(post_predictions, voxel_vols)]

In [10]:
# Initialize the VirtualRichard
virtual_richard = VirtualRichard()

In [11]:
# get the GT masks
gt_masks = [nib.load(path.replace("_support_analysis_10.nii.gz", "_seg.nii.gz")).get_fdata() for path in predictions_files]
post_gt_masks = postprocess_predictions(gt_masks)

100%|██████████| 93/93 [01:22<00:00,  1.13it/s]


In [12]:
# Get only the tumors that are bigger than 10 mm in diameter in the GT masks
gt_masks_big = [utils.mask_by_diameter(gt_mask, voxel_vol, 10)[1] for gt_mask, voxel_vol in zip(gt_masks, voxel_vols)]
post_gt_masks_big = postprocess_predictions(gt_masks_big)

100%|██████████| 93/93 [00:48<00:00,  1.93it/s]


In [13]:
detection_metrics = virtual_richard.evaluate_detection(post_predictions, post_gt_masks)

In [14]:
detection_metrics_big = virtual_richard.evaluate_detection(post_predictions_big, post_gt_masks_big)

In [16]:
segmentation_metrics = virtual_richard.evaluate_segmentation(post_predictions, post_gt_masks, variability_th)

In [18]:
segmentation_metrics_big = virtual_richard.evaluate_segmentation(post_predictions_big, post_gt_masks_big, variability_th)

#### Detection metrics

In [19]:
TP_score = pd.DataFrame(detection_metrics[0])
FP_score = pd.DataFrame(detection_metrics[1])
FN_score = pd.DataFrame(detection_metrics[2])

In [20]:
TP_score.describe()

,0
count,93.000000
mean,0.503875
std,0.264043
min,0.000000
25%,0.333333
50%,0.500000
75%,0.666667
max,1.000000


In [21]:
FP_score.describe()

,0
count,93.000000
mean,0.853351
std,0.135638
min,0.363636
25%,0.809524
50%,0.900000
75%,0.947368
max,1.000000


In [22]:
FN_score.describe()

,0
count,93.000000
mean,0.496125
std,0.264043
min,0.000000
25%,0.333333
50%,0.500000
75%,0.666667
max,1.000000


In [23]:
TP_score_big = pd.DataFrame(detection_metrics_big[0])
FP_score_big = pd.DataFrame(detection_metrics_big[1])
FN_score_big = pd.DataFrame(detection_metrics_big[2])

In [24]:
TP_score_big.describe()

,0
count,93.000000
mean,0.584137
std,0.317407
min,0.000000
25%,0.384615
50%,0.588235
75%,0.857143
max,1.000000


In [25]:
FP_score_big.describe()

,0
count,93.000000
mean,0.776354
std,0.204192
min,0.166667
25%,0.666667
50%,0.812500
75%,0.970588
max,1.000000


In [26]:
FN_score_big.describe()

,0
count,93.000000
mean,0.415863
std,0.317407
min,0.000000
25%,0.142857
50%,0.411765
75%,0.615385
max,1.000000


#### Segmentation metrics

In [33]:
contour_score =  pd.DataFrame(np.concatenate(segmentation_metrics[0]))
contour_per_slice_score = pd.DataFrame(np.concatenate(segmentation_metrics[1]))
contour_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big[0]))
contour_per_slice_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big[1]))

In [34]:
contour_score.describe()

,0
count,809.000000
mean,0.802956
std,0.298550
min,0.001367
25%,0.687648
50%,0.989081
75%,1.000000
max,1.000000


In [35]:
contour_score_big.describe()

,0
count,431.000000
mean,0.772205
std,0.302929
min,0.001262
25%,0.630805
50%,0.935725
75%,1.000000
max,1.000000


In [36]:
contour_per_slice_score.describe()

,0
count,2956.000000
mean,0.609725
std,0.276815
min,0.000000
25%,0.423398
50%,0.642459
75%,0.827070
max,1.000000


In [37]:
contour_per_slice_score_big.describe()

,0
count,2646.000000
mean,0.582892
std,0.279223
min,0.000000
25%,0.387522
50%,0.615385
75%,0.805482
max,1.000000


Let's check the results for Kuti's results

In [38]:
# Path to the folder with the scans and the corresponding predictions
predictions_files = sorted(glob(os.path.join(LIVER_LESIONS_DATASET, "*support_analysis_k10_ku.nii.gz")))
pattern = re.compile(r".*\d+_support_analysis_k10_ku.nii.gz")
predictions_files = [path for path in predictions_files if pattern.search(path)]
scans = [path.replace("support_analysis_k10_ku.nii.gz", "scan.nii.gz") for path in predictions_files]
liver_masks = [path.replace("support_analysis_k10_ku.nii.gz", "liver.nii.gz") for path in predictions_files]
# Load the scans and the predictions
scans = [nib.load(scan).get_fdata() for scan in scans]
predictions = [nib.load(prediction).get_fdata() for prediction in predictions_files]
liver_masks = [nib.load(liver_mask).get_fdata().astype("float64") for liver_mask in liver_masks]
# Get only predictions within the liver
predictions = [prediction * liver_mask for prediction, liver_mask in zip(predictions, liver_masks)]
# postprocess the predictions
post_predictions = postprocess_predictions(predictions, save_postprocessed=False,
                                           predictions_affines=[nib.load(path).affine for path in predictions_files],
                                           predictions_filenames=predictions_files)
# Get the voxel volume
voxel_vols = [np.prod(nib.load(path).header.get_zooms()) for path in predictions_files]
# Get tumor that are bigger than 10 mm in diameter
post_predictions_big = [utils.mask_by_diameter(prediction, voxel_vol, 10)[1] for prediction, voxel_vol in
                        zip(post_predictions, voxel_vols)]

100%|██████████| 93/93 [01:21<00:00,  1.14it/s]


In [40]:
detection_metrics = virtual_richard.evaluate_detection(post_predictions, post_gt_masks)
detection_metrics_big = virtual_richard.evaluate_detection(post_predictions_big, post_gt_masks_big)
segmentation_metrics = virtual_richard.evaluate_segmentation(post_predictions, post_gt_masks, variability_th)
segmentation_metrics_big = virtual_richard.evaluate_segmentation(post_predictions_big, post_gt_masks_big, variability_th)

In [41]:
TP_score = pd.DataFrame(detection_metrics[0])
FP_score = pd.DataFrame(detection_metrics[1])
FN_score = pd.DataFrame(detection_metrics[2])
TP_score_big = pd.DataFrame(detection_metrics_big[0])
FP_score_big = pd.DataFrame(detection_metrics_big[1])
FN_score_big = pd.DataFrame(detection_metrics_big[2])

In [42]:
TP_score.describe()

,0
count,93.000000
mean,0.432443
std,0.298248
min,0.000000
25%,0.181818
50%,0.418605
75%,0.611111
max,1.000000


In [43]:
FP_score.describe()

,0
count,93.000000
mean,0.744367
std,0.216507
min,0.148148
25%,0.580000
50%,0.814815
75%,0.909091
max,1.000000


In [44]:
FN_score.describe()

,0
count,93.000000
mean,0.567557
std,0.298248
min,0.000000
25%,0.388889
50%,0.581395
75%,0.818182
max,1.000000


In [45]:
TP_score_big.describe()

,0
count,93.000000
mean,0.532863
std,0.334726
min,0.000000
25%,0.312500
50%,0.500000
75%,0.818182
max,1.000000


In [46]:
FP_score_big.describe()

,0
count,93.000000
mean,0.496017
std,0.358058
min,0.000000
25%,0.200000
50%,0.478261
75%,0.790698
max,1.000000


In [47]:
FN_score_big.describe()

,0
count,93.000000
mean,0.467137
std,0.334726
min,0.000000
25%,0.181818
50%,0.500000
75%,0.687500
max,1.000000


In [48]:
contour_score =  pd.DataFrame(np.concatenate(segmentation_metrics[0]))
contour_per_slice_score = pd.DataFrame(np.concatenate(segmentation_metrics[1]))
contour_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big[0]))
contour_per_slice_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big[1]))

In [49]:
contour_score.describe()

,0
count,676.000000
mean,0.741900
std,0.320122
min,0.000230
25%,0.549790
50%,0.906441
75%,1.000000
max,1.000000


In [50]:
contour_score_big.describe()

,0
count,370.000000
mean,0.709378
std,0.315199
min,0.003853
25%,0.488174
50%,0.844501
75%,0.997869
max,1.000000


In [51]:
contour_per_slice_score.describe()

,0
count,2572.000000
mean,0.515906
std,0.307361
min,0.000000
25%,0.266317
50%,0.524107
75%,0.759680
max,1.000000


In [52]:
contour_per_slice_score_big.describe()

,0
count,2235.000000
mean,0.488226
std,0.301202
min,0.000000
25%,0.235905
50%,0.487979
75%,0.715175
max,1.000000


Let's check the results for the best clustering filtering model predictions

In [11]:
# Path to the folder with the scans and the corresponding predictions
predictions_files = sorted(glob(os.path.join(LIVER_LESIONS_DATASET, "*support_analysis_100_clusters_k10.nii.gz")))
pattern = re.compile(r".*\d+_support_analysis_100_clusters_k10.nii.gz")
predictions_files = [path for path in predictions_files if pattern.search(path)]
scans = [path.replace("support_analysis_100_clusters_k10.nii.gz", "scan.nii.gz") for path in predictions_files]
liver_masks = [path.replace("support_analysis_100_clusters_k10.nii.gz", "liver.nii.gz") for path in predictions_files]
# Load the scans and the predictions
scans = [nib.load(scan).get_fdata() for scan in scans]
predictions = [nib.load(prediction).get_fdata() for prediction in predictions_files]
liver_masks = [nib.load(liver_mask).get_fdata().astype("float64") for liver_mask in liver_masks]
# Get only predictions within the liver
predictions = [prediction * liver_mask for prediction, liver_mask in zip(predictions, liver_masks)]


In [50]:
# postprocess the predictions
post_predictions = postprocess_predictions(predictions, save_postprocessed=False,
                                           predictions_affines=[nib.load(path).affine for path in predictions_files],
                                           predictions_filenames=predictions_files)
# # Get the voxel volume
voxel_vols = [np.prod(nib.load(path).header.get_zooms()) for path in predictions_files]
# Get tumor that are bigger than 10 mm in diameter
post_predictions_big = [utils.mask_by_diameter(prediction, voxel_vol, 10)[1] for prediction, voxel_vol in
                        tqdm(zip(post_predictions, voxel_vols), total=len(post_predictions))]

100%|██████████| 93/93 [01:41<00:00,  1.09s/it]


In [13]:
virtual_richard = VirtualRichard()

In [38]:
# Get the GT masks
gt_masks = [nib.load(path.replace("support_analysis_100_clusters_k10.nii.gz", "seg.nii.gz")).get_fdata() for path in predictions_files]
post_gt_masks = postprocess_predictions(gt_masks)
# Get only the tumors that are bigger than 10 mm in diameter in the GT masks
gt_masks_big = [utils.mask_by_diameter(gt_mask, voxel_vol, 10)[1] for gt_mask, voxel_vol in tqdm(zip(gt_masks, voxel_vols))]
post_gt_masks_big = postprocess_predictions(gt_masks_big)

100%|██████████| 93/93 [00:46<00:00,  2.00it/s]


In [44]:
from medpy import metric
recalls_small = [metric.recall(p, g) for p, g in zip(post_predictions, post_gt_masks)]
pd.DataFrame(recalls_small).describe()

,0
count,93.000000
mean,0.312230
std,0.208190
min,0.000000
25%,0.133023
50%,0.336521
75%,0.470563
max,0.741287


In [58]:
metric.obj_tpr(post_predictions_big[32], post_gt_masks_big[32])

0.8333333333333334

In [47]:
tpr_small = [metric.obj_tpr(p, g) for p, g in zip(post_predictions, post_gt_masks)]
pd.DataFrame(tpr_small).describe()

,0
count,93.000000
mean,0.263772
std,0.223317
min,0.000000
25%,0.103175
50%,0.200000
75%,0.400000
max,1.000000


In [45]:
recalls_big = [metric.recall(p, g) for p, g in zip(post_predictions_big, post_gt_masks_big)]
pd.DataFrame(recalls_big).describe()

,0
count,93.000000
mean,0.296958
std,0.231386
min,0.000000
25%,0.057543
50%,0.306717
75%,0.463702
max,0.862958


In [59]:
detection_metrics = virtual_richard.evaluate_detection(post_predictions, post_gt_masks)
detection_metrics_big = virtual_richard.evaluate_detection(post_predictions_big, post_gt_masks_big)

In [68]:
segmentation_metrics = virtual_richard.evaluate_segmentation(post_predictions, post_gt_masks, variability_th)
segmentation_metrics_big = virtual_richard.evaluate_segmentation(post_predictions_big, post_gt_masks_big, variability_th)

In [61]:
TP_score = pd.DataFrame(detection_metrics[0])
FP_score = pd.DataFrame(detection_metrics[1])
FN_score = pd.DataFrame(detection_metrics[2])
TP_score_big = pd.DataFrame(detection_metrics_big[0])
FP_score_big = pd.DataFrame(detection_metrics_big[1])
FN_score_big = pd.DataFrame(detection_metrics_big[2])

In [62]:
TP_score.describe()

,0
count,93.000000
mean,0.493118
std,0.287417
min,0.000000
25%,0.285714
50%,0.500000
75%,0.666667
max,1.000000


In [63]:
FP_score.describe()

,0
count,93.000000
mean,0.736228
std,0.223317
min,0.000000
25%,0.600000
50%,0.800000
75%,0.896825
max,1.000000


In [64]:
FN_score.describe()

,0
count,93.000000
mean,0.506882
std,0.287417
min,0.000000
25%,0.333333
50%,0.500000
75%,0.714286
max,1.000000


In [65]:
TP_score_big.describe()

,0
count,93.000000
mean,0.614343
std,0.327055
min,0.000000
25%,0.411765
50%,0.625000
75%,1.000000
max,1.000000


In [66]:
FP_score_big.describe()

,0
count,93.000000
mean,0.524015
std,0.350302
min,0.000000
25%,0.222222
50%,0.555556
75%,0.830189
max,1.000000


In [67]:
FN_score_big.describe()

,0
count,93.000000
mean,0.385657
std,0.327055
min,0.000000
25%,0.000000
50%,0.375000
75%,0.588235
max,1.000000


In [69]:
contour_score =  pd.DataFrame(np.concatenate(segmentation_metrics[0]))
contour_per_slice_score = pd.DataFrame(np.concatenate(segmentation_metrics[1]))
contour_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big[0]))
contour_per_slice_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big[1]))

In [70]:
contour_score.describe()

,0
count,810.000000
mean,0.799442
std,0.298621
min,0.000502
25%,0.699510
50%,0.983632
75%,1.000000
max,1.000000


In [71]:
contour_per_slice_score.describe()

,0
count,2790.000000
mean,0.589275
std,0.316820
min,0.000000
25%,0.346488
50%,0.621463
75%,0.872435
max,1.000000


In [72]:
contour_score_big.describe()

,0
count,407.000000
mean,0.745567
std,0.309313
min,0.000000
25%,0.538495
50%,0.909457
75%,1.000000
max,1.000000


In [73]:
contour_per_slice_score_big.describe()

,0
count,2412.000000
mean,0.543688
std,0.315561
min,0.000000
25%,0.287532
50%,0.555974
75%,0.816320
max,1.000000
